In [1]:
import os
import json
import random
import functools
import copy
from typing import Optional, List

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset, TensorDataset

# Pyro (Probabilistic Programming)
import pyro
from pyro.infer import SVI, TraceMeanField_ELBO, Trace_ELBO
from pyro.nn.module import to_pyro_module_

# TyXe (Bayesian Neural Networks)
import tyxe

# Hugging Face Transformers and PEFT (Parameter-Efficient Fine-Tuning)
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import (
    PeftConfig,
    PeftModel,
    LoraConfig,
    get_peft_model,
)
from peft.tuners.lora import LoraLayer

# Hugging Face Datasets
from datasets import Dataset, load_dataset  # Dataset creation and SuperNI dataset

# Accelerate (Efficient Distributed Training)
from accelerate import init_empty_weights

# Hugging Face Hub
from huggingface_hub import login

# BitsAndBytes (Optional: Quantization Optimization)
import bitsandbytes

# NumPy
import numpy as np


In [2]:
import torch

torch.cuda.set_device(2)  # Set to GPU 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


if torch.cuda.is_available():
    current_device = torch.cuda.current_device()  # Get the current CUDA device index
    device_name = torch.cuda.get_device_name(current_device)  # Get the device name
    print(f"CUDA is currently using device {current_device}: {device_name}")
else:
    print("CUDA is not available.")

CUDA is currently using device 2: NVIDIA L40S


In [4]:
os.chdir('/home/pranav24/cs-546/SSR/Latest_Weights/QA_QG_Weights')
target_file = "task1312_amazonreview_polarity_classification.json"

with open(target_file, 'r', encoding='utf-8-sig') as f:
    json_data = json.load(f)
from transformers import AutoTokenizer

# Replace 'path/to/seed-model' with your seed model's identifier or local path
seed_model_path = "/home/pranav24/cs-546/Llama-3.2-3B/finetuned_Lora/QA_Weights"
tokenizer = AutoTokenizer.from_pretrained(seed_model_path)

instances = json_data['Instances'][4500:5000]
instruct1="You will be given a sentence describing an experience. You need to classify its sentiment, which could only be either positive or negative. \nExperience: "
instruct2="\nSentiment: "
input_texts = [str(instruct1+instance['input']+instruct2) for instance in instances]
output_texts = [str(instance['output'][0]) if instance['output'] else "" for instance in instances]

print(input_texts[0])
print(output_texts[0])

# Create Hugging Face Dataset
ds = Dataset.from_dict({'input': input_texts, 'output': output_texts})

# Tokenize the dataset
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["output"],
            truncation=True,
            padding="max_length",
            max_length=512
        )
    model_inputs["labels"] = labels["input_ids"]
    model_inputs["attention_mask"] = model_inputs.get("attention_mask", None)
    return model_inputs

# Apply tokenization and set format
tokenized_datasets = ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])
tokenized_datasets.set_format("torch")

# Split dataset into train and eval
train_size = int(0.8 * len(tokenized_datasets))
train_dataset = tokenized_datasets.select(range(train_size))
eval_dataset = tokenized_datasets.select(range(train_size, len(tokenized_datasets)))

# Create DataLoaders
batch_size = 8  
train_loader_2 = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
eval_loader_2 = DataLoader(eval_dataset, batch_size=batch_size)

# Define data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

You will be given a sentence describing an experience. You need to classify its sentiment, which could only be either positive or negative. 
Experience: I ordered from Amazon because the only 5oz cups locally seem to be child oriented and not for an adult bathroom. Ha! My dixie cups have riddles on them and are as cartoonish as the ones I could have bought down the street. I'm throwing them out and getting a hard plastic cup. So mean to send something other than was pictured!!!
Sentiment: 
negative


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
import json_repair 
os.chdir('/home/pranav24/cs-546/Llama-3.2-3B/SSR')
target_file = "final_sampled.jsonl"

with open(target_file, 'r', encoding='utf-8-sig') as f:
    json_data = json_repair.loads(f.read())

instances = json_data
input_texts = ["\nContext: "+instance['context']+ "\nQuestion: " + instance['question'] for instance in instances]
output_texts = [instance["refined_answer"] for instance in instances]


ds = Dataset.from_dict({'input': input_texts, 'output': output_texts})
tokenized_datasets = ds.map(tokenize_function, batched=True, remove_columns=["input", "output"])
tokenized_datasets.set_format("torch")
train_size = int(1.0 * len(tokenized_datasets))
synthetic_train_dataset = tokenized_datasets.select(range(train_size))
batch_size = 8  
synthetic_loader_1 = DataLoader(synthetic_train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
from torch.utils.data import ConcatDataset, DataLoader

# Combine datasets
if synthetic_loader_1 is not None:
    print('combined dataloader')
    combined_dataset = ConcatDataset([train_loader_2.dataset, synthetic_loader_1.dataset])
    combined_loader = DataLoader(combined_dataset, batch_size=8, shuffle=True)
else:
    print('not combined dataloader')
    combined_loader = train_loader_2

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig
from torch.utils.data import DataLoader

# Set environment variable to manage memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Directories for saving model and offloading
save_dir = os.path.expanduser("fine_tuned_sa_noEVCL2/")
offload_dir = os.path.expanduser("offload_folder/")
os.makedirs(save_dir, exist_ok=True)
os.makedirs(offload_dir, exist_ok=True)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(seed_model_path)  # Use the same tokenizer as your QA model
tokenizer.pad_token = tokenizer.eos_token

# Configure bitsandbytes quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True  # Enable FP32 offloading for CPU
)
# Load the QA fine-tuned model
model = AutoModelForCausalLM.from_pretrained(
    seed_model_path,
    device_map="auto",  # Automatically distribute layers across devices
    offload_folder=offload_dir,  # Specify offloading directory for disk storage
    quantization_config=quantization_config,
)

# Configure LoRA for fine-tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Use the combined loader dataset for training
train_dataset = combined_loader.dataset  # Dataset defined in your earlier code

# Training arguments
training_args = TrainingArguments(
    output_dir=save_dir,
    logging_steps=500,
    save_steps=1000,
    save_total_limit=2,
    per_device_train_batch_size=8,  # Batch size of 8
    gradient_accumulation_steps=4,  # Adjust based on memory constraints
    num_train_epochs=3,           # Number of epochs
    learning_rate=2e-4,             # Learning rate
    fp16=torch.cuda.is_available(), # Enable mixed precision for GPUs
    report_to="none",
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Resume training from checkpoint if available
checkpoint_path = os.path.join(save_dir, "checkpoint-latest")
if os.path.exists(checkpoint_path):
    print(f"Resuming training from checkpoint: {checkpoint_path}")
    trainer.train(resume_from_checkpoint=checkpoint_path)
else:
    trainer.train()

# Save the fine-tuned model and tokenizer
model.save_pretrained(os.path.join(save_dir, "fine-tuned-sa-lora_noEVCL2"))
tokenizer.save_pretrained(os.path.join(save_dir, "fine-tuned-sa-lora_noEVCL2"))

print("Fine-tuning complete. Model and tokenizer saved!")
